### Алгоритм роя частиц. Оптимизация гиперпараметров при помощи PySwarm

In [46]:
import numpy as np
import pandas as pd
import time
import pyswarms as ps

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

В генетическом алгоритме особи образуют поколения, среди которых выбираюьтся лучше, а в алгоритме роя частиц точки запоминают свои лучшие значения и к ним стремятся.

### KNeighbors Classifier

Создадим рой из 40 частиц и проведем 5 итераций алгоритма. Минимальное число соседей частицы - 3, максимальное - 30. Используем городскую или евклидову метрики. Выбираем для весов `uniform` или `distance`. Задаем через `numpy arrays`, так как `pyswars` работает с массивами чисел, а не со словарями, как `pygad`.

In [47]:
n_particles = 40
iters = 5
total = n_particles * iters

bounds_knn = (np.array([3, 1, 0]), np.array([30, 5, 2]))

Обработка каждой частицы.

In [48]:
def fitness_func_knn(matrix, **kwargs):
    global x_train, y_train
    
    scores = []
    for i in range(n_particles):
        weight_choice = 'uniform' if matrix[i, 2] < 1 else 'distance'

        model = KNeighborsClassifier(
            n_neighbors=int(matrix[i][0]),
            p=int(matrix[i][1]),
            weights=weight_choice,
            n_jobs=-1       # использование всех доступных потоков
        )

        score = cross_val_score(
            model,
            x_train,
            y_train,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1
        ).mean()        # среднее, тк по всем вариантам 

        scores.append(score)

    return 1 - np.array(scores)     # pso ищет min зн-е ф-ии, поэтому считаем ошибку

Настройка PSO алгоритма.

In [49]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results_knn = []

In [50]:
params = {
    'c1' : 0.5,     # насколько частицу тянет к её собственному самому лучшему результату
    'c2' : 0.3,     # насколько сильно частицу тянет к общему лидеру
    'w' : 0.9       # желание частицы лететь в том же направлении
}

In [ ]:
for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"../classical_ml_methods/f1_data/f1_data_{n}_s_{m}_f.csv")

        y = data['collision']
        x = data.drop(['collision'], axis=1)

        global x_train, y_train
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.2, random_state=81
            )
        
        optimizer = ps.single.GlobalBestPSO(        # мало гиперпараметров, не бинарные
            n_particles=n_particles,
            dimensions=3,
            options=params,
            bounds=bounds_knn
        )

        start_time = time.time()
        best_error_knn, best_pos_knn = optimizer.optimize(    
            fitness_func_knn, 
            iters=iters
        )
        search_time = time.time() - start_time

        results_knn.append({
            'samples (n)': n,
            'features (m)': m,
            'best_n_neighbors': int(best_pos_knn[0]),
            'best_p': int(best_pos_knn[1]),
            'best_weight' : 'uniform' if best_pos_knn[2] < 1 else 'distance',
            'f1-score': 1 - best_error_knn,
            'search_time (sec)': search_time
        })


2026-03-13 18:54:45,605 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0      
2026-03-13 18:54:52,843 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0, best pos: [18.61661801  2.63557421  0.76790425]
2026-03-13 18:54:52,847 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0
2026-03-13 18:54:58,007 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0, best pos: [22.09943633  3.41604455  0.83259713]
2026-03-13 18:54:58,012 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.156
2026-03-13 18:55:03,184 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.155548522659123, best pos: [3.693043

In [52]:
results_knn_df = pd.DataFrame(results_knn)
results_knn_df

,samples (n),features (m),best_n_neighbors,best_p,best_weight,f1-score,search_time (sec)
0,100,5,18,2,uniform,1.000000,7.237400
1,100,8,22,3,uniform,1.000000,5.160410
2,100,11,3,2,uniform,0.844451,5.172054
3,500,5,16,2,uniform,0.992776,5.329554
4,500,8,12,2,distance,0.982767,5.440877
5,500,11,4,1,distance,0.832888,5.692319
6,1000,5,8,3,uniform,0.996364,5.924589
7,1000,8,20,2,uniform,0.984668,5.859598
8,1000,11,4,1,distance,0.837058,6.341459
9,3000,5,6,1,distance,0.995895,6.433576


**Сравнение с предыдущими результатами:**
- `f1-score` на больших выборках увеличился. Заметим, что немного ниже, чем при использовании `pygad`, однако времени на подбор параметров PSO требует больше
- в среднем, значения `n_neighbors` увеличились по сравнению с `pygad` и предыдущей лабораторной работой
- алгоритм роя частиц потребовал больше времени, нежели `RandomizedSearchCV` или `pygad`
- почти всегда для выбирается `distance` для учета мнения "соседей"

![KNN before](knn.png)

### Сохранение модели

In [58]:
import joblib

model = KNeighborsClassifier(
    n_neighbors=int(best_pos_knn[0]),
    p=int(best_pos_knn[1]),
    weights='uniform' if best_pos_knn[2] < 1 else 'distance'
)
model.fit(x_train, y_train) 

joblib.dump(model, 'models/knn_pso_model.pkl')

['models/knn_pso_model.pkl']

### Native Bayes

Возьмем достаточно маленький параметр `var_smoothing`, отвечающий за «сглаживание» кривой распределения, чтобы модель не выдавала нулевую вероятность для данных, которые не встречались при обучении.

In [57]:
bounds_nb = (np.array([1e-10]), np.array([10e-7]))

In [62]:
def fitness_func_nb(matrix, **kwargs):
    global x_train, y_train
    
    scores = []
    for i in range(n_particles):

        model = GaussianNB(
            var_smoothing=matrix[i][0],
            #n_jobs=-1      
        )

        score = cross_val_score(
            model,
            x_train,
            y_train,
            cv=5,
            scoring='f1_macro',
            n_jobs=-1
        ).mean()        

        scores.append(score)

    return 1 - np.array(scores)     

In [59]:
results_nb = []

In [ ]:
for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"../classical_ml_methods/f1_data/f1_data_{n}_s_{m}_f.csv")

        y = data['collision']
        x = data.drop(['collision'], axis=1)

        global x_train, y_train
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.2, random_state=81
            )
        
        optimizer = ps.single.GlobalBestPSO(        
            n_particles=n_particles,
            dimensions=1,
            options=params,
            bounds=bounds_nb
        )

        start_time = time.time()
        best_error_nb, best_pos_nb = optimizer.optimize(    
            fitness_func_nb, 
            iters=iters
        )
        search_time = time.time() - start_time

        results_nb.append({
            'samples (n)': n,
            'features (m)': m,
            'best_var_smoothing': best_pos_nb[0],
            'f1-score': 1 - best_error_nb,
            'search_time (sec)': search_time
        })

2026-03-13 19:10:51,241 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.0296
2026-03-13 19:10:53,875 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.029573934837092808, best pos: [7.57797883e-07]
2026-03-13 19:10:53,879 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.0217
2026-03-13 19:10:56,478 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.021671826625387025, best pos: [4.68386902e-07]
2026-03-13 19:10:56,482 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|5/5, best_cost=0.08
2026-03-13 19:10:59,119 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.07998568481327106, best pos: [9.82855052e

In [67]:
results_nb_df = pd.DataFrame(results_nb)
results_nb_df

,samples (n),features (m),best_var_smoothing,f1-score,search_time (sec)
0,100,5,7.577979e-07,0.970426,2.634373
1,100,8,4.683869e-07,0.978328,2.598744
2,100,11,9.828551e-07,0.920014,2.637649
3,500,5,6.549071e-07,0.976426,2.651890
4,500,8,2.141311e-07,0.905915,2.628057
5,500,11,7.465358e-07,0.878099,2.654647
6,1000,5,1.302806e-07,0.987269,2.645138
7,1000,8,4.288515e-08,0.968370,2.644840
8,1000,11,9.110327e-07,0.874572,2.640900
9,3000,5,3.448016e-07,0.985785,2.785177


**Сравнение с предыдущими результатами:**
- `f1-score` незначительно уменьшился. Заметим, что при использовании генетического алгоритма получили такие же значения, но немного быстрее
- `var_smoothing` изменили свои значения
- алгоритм роя частиц потребовал больше времени, нежели `RandomizedSearchCV`

![NB before](native_bayes.png)

### Сохранение модели

In [ ]:
model = GaussianNB(
    var_smoothing=best_pos_nb[0]
)
model.fit(x_train, y_train) 

joblib.dump(model, 'models/native_bayes_pso_model.pkl')